# Cellpose evaluation: original vs. Claude-edited weights

This notebook scores the segmentations of **Cellpose cyto3** and **Cellpose-SAM**, and of their
weight-space-edited versions (`*_CLAUDE`), against the ground truth in `ground_truth/cellpose/`.
It covers three datasets: `test` (68 images), `train` (540) and `train_cyto2` (256).

* **Inputs.** The notebook reads the predicted masks wherever the editing sandbox wrote them; the
  pointers are set in **Configuration**. It never writes into a sandbox. Tables go to `results/`
  and figures to `figures/`.
* **Metrics.** As in the Cellpose-SAM paper (`paper/cpsam/benchmarks.py`, `figures.py`), the
  metric code is the sandbox repo's own `cellpose.metrics.average_precision`, at IoU 0.50–0.95.
  AP = TP / (TP + FP + FN) and error rate = (FP + FN) / (TP + FN), both per image. Original and
  edited models are compared with a paired two-sided Wilcoxon signed-rank test.
* **Held-out images.** An image an editing session looked at while designing its edits (its `dev`
  split) is not held out for any model. The figures are drawn for held-out images and for all
  images (`SUBSETS`).
* **Headline dataset.** `train` and `train_cyto2` are training images for both model families, so
  the main result is `test`.

Kernel: **Python (Pseudochannel)**. It has matplotlib, scipy, cv2 and fastremap. The Cellpose code
itself is imported from the sandbox repo.

In [ ]:
import sys
from pathlib import Path

# The editing sandbox (Cellpose repo, weights, eval images, predictions). READ-ONLY from here
# (CLAUDE.md rule 1): this notebook only reads from it.
SANDBOX = Path("/mnt/Vol_b/cellpose")
sys.dont_write_bytecode = True          # importing from the sandbox must not create __pycache__ there
sys.path.insert(0, str(SANDBOX))        # use the sandbox repo's own metric code

import hashlib
import json
import warnings

import cv2
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from PIL import Image
from scipy.ndimage import find_objects, uniform_filter
from scipy.optimize import linear_sum_assignment
from scipy.stats import wilcoxon

from cellpose import metrics

assert Path(metrics.__file__).resolve().is_relative_to(SANDBOX), f"wrong cellpose: {metrics.__file__}"

REPO = Path.cwd()
assert (REPO / "CLAUDE.md").is_file(), "start the notebook from the repo root"
GT_ROOT = REPO / "ground_truth" / "cellpose"
IMAGE_ROOT = SANDBOX / "data"           # <dataset>/<id>_img.png, only used for display
RESULTS, FIGURES = REPO / "results", REPO / "figures"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

# figure style of the Cellpose-SAM paper (paper/cpsam/fig_utils.py)
mpl.rcParams.update({
    "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlelocation": "left", "axes.titleweight": "normal",
    "savefig.dpi": 300, "savefig.bbox": "tight", "pdf.fonttype": 42, "ps.fonttype": 42,
})
if any(f.name == "Arial" for f in font_manager.fontManager.ttflist):
    mpl.rcParams["font.family"] = "Arial"
pd.set_option("display.width", 200)
print("cellpose.metrics from", metrics.__file__)

## Configuration

Each entry in `MODELS` holds **pointers** to one model's output. For each dataset, `sources` lists
path templates, which are tried in order; the first existing file is used.

* `{id}` is the three-digit image number, as in `000_img.png` / `000_masks.png`.
* `{dataset}` is the dataset name.
* Every file must be a uint16 instance-label image (0 = background) of the same size as the
  ground truth. Cellpose's `*_cp_masks.png` files have this format.

`sandbox_preds(...)` builds the templates for the editing harness's own folders
(`preds/<run>/inst/<dataset>__<id>_img.png`). A model needs several runs when its dev and check
images were predicted separately.

**cyto3:** the cyto3 editing session (`weight_space_work_cyto3`) is still running. Its test-set
exports and its edited predictions don't exist yet. Point the two `TODO` entries at them once they
do. Until then, cyto3 shows up only in the coverage table.

In [ ]:
DATASETS = {                      # ground_truth/cellpose/<name>/  ->  label in figures
    "test":        "Cellpose test",
    "train":       "Cellpose train",
    "train_cyto2": "cyto2 train",
}
IOU_THRESHOLDS = np.round(np.arange(0.5, 1.0, 0.05), 2)   # as in paper/cpsam/benchmarks.py
IOU_MAIN = 0.5                                            # IoU of the violin plots (paper: 0.5)


def sandbox_preds(work, *runs):
    """Templates for <work>/preds/<run>/inst/<dataset>__<id>_img.png, searched in the given order."""
    return [str(work / "preds" / run / "inst" / "{dataset}__{id}_img.png") for run in runs]


WS_CPSAM = SANDBOX / "weight_space_work"
WS_CYTO3 = SANDBOX / "weight_space_work_cyto3"

# Colours: the paper's family hues (cyto3 green, Cellpose-SAM purple), with a darker step of the same
# hue for the Claude-edited weights. Checked with the dataviz palette validator (light surface: lightness
# band, chroma, CVD and normal-vision separation, contrast all pass). Edited models are also hatched.
MODELS = {
    "cyto3": dict(
        label="cyto3", family="cyto3", edited=False, color="#3ea53e",
        weights=SANDBOX / "weights/cyto3",
        sources={
            "test":        [],                                   # TODO: cyto3 test-set export
            "train":       sandbox_preds(WS_CYTO3, "baseline"),  # so far only the 24 dev images
            "train_cyto2": sandbox_preds(WS_CYTO3, "baseline"),
        }),
    "cyto3_CLAUDE": dict(
        label="cyto3 + Claude", family="cyto3", edited=True, color="#1c6a2a",
        weights=SANDBOX / "weights/cyto3_CLAUDE",
        sources={"test": [], "train": [], "train_cyto2": []}),  # TODO: cyto3 session not finished
    "cpsam": dict(
        label="Cellpose-SAM", family="cpsam", edited=False, color="#ac70f5",
        weights=SANDBOX / "weights/cpsam_v2",
        sources={
            "test":        [str(SANDBOX / "data/outputs/Original/{id}_img_cp_masks.png")],
            "train":       sandbox_preds(WS_CPSAM, "baseline", "baseline_check"),
            "train_cyto2": sandbox_preds(WS_CPSAM, "baseline", "baseline_check"),
        }),
    "cpsam_CLAUDE": dict(
        label="Cellpose-SAM + Claude", family="cpsam", edited=True, color="#5f30b5",
        weights=SANDBOX / "weights/cpsam_v2_CLAUDE",
        sources={
            "test":        [str(SANDBOX / "data/outputs/CLAUDE/{id}_img_cp_masks.png")],
            "train":       sandbox_preds(WS_CPSAM, "e001", "final_check"),
            "train_cyto2": sandbox_preds(WS_CPSAM, "e001", "final_check"),
        }),
}

# Per model family: the session's dev split(s) (images looked at while designing edits) and its
# integrity record (weights hashes).
FAMILIES = {
    "cyto3": dict(dev_lists=sorted((WS_CYTO3 / "splits").glob("dev*.txt")), integrity=WS_CYTO3 / "integrity.json"),
    "cpsam": dict(dev_lists=sorted((WS_CPSAM / "splits").glob("dev*.txt")), integrity=WS_CPSAM / "integrity.json"),
}

SUBSETS = ["held_out", "all"]   # one violin figure each; held_out = in no session's dev split
REQUIRE_COMPLETE = True         # score a model on a dataset only if it has a prediction for every image
HASH_WEIGHTS = True             # SHA-256 the weight files for the provenance check (~10 s)
GT_BACKGROUND_65535 = True      # some GT files store background as 65535 (-1) and use 0 for a cell; see Inventory

## Inventory: ground truth, dev splits, available predictions

In [ ]:
def read_labels(path):
    """uint16 label PNG -> int32 (uint16 overflows inside cellpose.metrics when a label is 65535)."""
    a = np.asarray(Image.open(path))
    assert a.ndim == 2 and np.issubdtype(a.dtype, np.integer), f"{path}: not a 2-D label image ({a.dtype}, {a.shape})"
    return a.astype(np.int32)


def read_gt(path):
    """Ground truth with background = 0. A few files encode background as 65535 (-1 written to uint16)
    and give one cell the label 0; with GT_BACKGROUND_65535 they are remapped (cell k -> k+1, 65535 -> 0)."""
    a = read_labels(path)
    if GT_BACKGROUND_65535 and a.max() == 65535:
        a = a + 1
        a[a == 65536] = 0
    return a


# ground truth
rows = []
for ds in DATASETS:
    files = sorted((GT_ROOT / ds).glob("*_masks.png"))
    assert files, f"no ground truth in {GT_ROOT / ds}"
    rows += [dict(dataset=ds, id=f.name.split("_")[0], gt=str(f)) for f in files]
IMAGES = pd.DataFrame(rows)

# GT files whose background is 65535 instead of 0
odd = []
for r in IMAGES.itertuples():
    a = read_labels(r.gt)
    if a.max() == 65535:
        odd.append(dict(file=f"{r.dataset}/{Path(r.gt).name}", background_65535=np.mean(a == 65535),
                        label0_pixels=int((a == 0).sum()), cells=len(np.unique(a)) - 1))
if odd:
    print(f"{len(odd)} GT files use 65535 as background (and 0 as a cell label); "
          f"{'remapped' if GT_BACKGROUND_65535 else 'NOT remapped (GT_BACKGROUND_65535=False)'}:")
    display(pd.DataFrame(odd).round(3))

# images each editing session looked at while designing its edits
for fam, cfg in FAMILIES.items():
    seen = set()
    for f in cfg["dev_lists"]:
        for line in Path(f).read_text().split():
            ds, name = line.split("/")
            seen.add((ds, name.split("_")[0]))
    IMAGES[f"dev_{fam}"] = [(d, i) in seen for d, i in zip(IMAGES.dataset, IMAGES.id)]
    print(f"{fam:6s} dev split: {[Path(f).name for f in cfg['dev_lists']]} -> {len(seen)} images")
IMAGES["held_out"] = ~IMAGES[[f"dev_{fam}" for fam in FAMILIES]].any(axis=1)


# predictions
def resolve(model, dataset, id_):
    for tpl in MODELS[model]["sources"].get(dataset, []):
        p = Path(tpl.format(dataset=dataset, id=id_))
        if p.is_file():
            return str(p)
    return None


for m in MODELS:
    IMAGES[m] = pd.Series([resolve(m, d, i) for d, i in zip(IMAGES.dataset, IMAGES.id)], dtype=object)


def has_pred(x):
    return isinstance(x, str)


def n_found(m, ds):
    return int(IMAGES.loc[IMAGES.dataset == ds, m].notna().sum())


N_GT = IMAGES.groupby("dataset").size()
INCLUDED = {ds: [m for m in MODELS
                if (n_found(m, ds) == N_GT[ds] if REQUIRE_COMPLETE else n_found(m, ds) > 0)]
           for ds in DATASETS}

coverage = pd.DataFrame(
    {ds: [f"{n_found(m, ds)}/{N_GT[ds]}" for m in MODELS] for ds in DATASETS},
    index=[MODELS[m]["label"] for m in MODELS])
coverage.loc["held-out images"] = [f"{IMAGES.loc[IMAGES.dataset == ds, 'held_out'].sum()}/{N_GT[ds]}" for ds in DATASETS]
print("\npredictions found per dataset (with REQUIRE_COMPLETE, models below the full count are not scored):")
display(coverage)
for ds, ms in INCLUDED.items():
    print(f"  {ds:12s} scored: {[MODELS[m]['label'] for m in ms]}")

### Provenance check

Before scoring (CLAUDE.md rule 5), this cell checks the following for every folder that supplied
predictions:

* the weights the folder says it was made with (`manifest.json` or `WEIGHTS.txt`) are the
  configured weights;
* every prediction is newer than the weight file, so no prediction comes from an earlier edit
  state;
* the weight file's SHA-256 on disk, and the one recorded in the manifest, match the session's
  `integrity.json`.

`None` means the check couldn't be made. One example is an edited checkpoint that the session
hasn't verified yet.

In [ ]:
def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def recorded_by(folder):
    """(weights path, sha256 or None) that a prediction folder records about itself."""
    for d in (folder, folder.parent):
        if (d / "manifest.json").is_file():
            man = json.loads((d / "manifest.json").read_text())
            return man.get("weights"), man.get("weights_sha256")
        if (d / "WEIGHTS.txt").is_file():
            return (d / "WEIGHTS.txt").read_text().strip(), None
    return None, None


rows = []
for m, cfg in MODELS.items():
    files = [Path(f) for f in IMAGES[m].dropna()]
    if not files:
        rows.append(dict(model=cfg["label"], folder="(no predictions found)"))
        continue
    integ_path = FAMILIES[cfg["family"]]["integrity"]
    integ = json.loads(integ_path.read_text()) if integ_path.is_file() else {}
    expected = integ.get("verified", {}).get("claude_sha256") if cfg["edited"] else integ.get("original_sha256")
    on_disk = sha256(cfg["weights"]) if HASH_WEIGHTS else None
    w_mtime = cfg["weights"].stat().st_mtime
    for folder in sorted({f.parent for f in files}):
        rec_w, rec_sha = recorded_by(folder)
        in_folder = [f for f in files if f.parent == folder]
        rows.append(dict(
            model=cfg["label"], folder=str(folder.relative_to(SANDBOX)), n=len(in_folder),
            recorded_weights=Path(rec_w).name if rec_w else None,
            weights_match=Path(rec_w).resolve() == cfg["weights"].resolve() if rec_w else None,
            newer_than_weights=min(f.stat().st_mtime for f in in_folder) >= w_mtime,
            sha_disk_eq_integrity=None if expected is None or on_disk is None else on_disk == expected,
            sha_manifest_eq_integrity=None if expected is None or rec_sha is None else rec_sha == expected,
        ))
PROVENANCE = pd.DataFrame(rows)
display(PROVENANCE)
checks = ["weights_match", "newer_than_weights", "sha_disk_eq_integrity", "sha_manifest_eq_integrity"]
bad = PROVENANCE[[c for c in checks if c in PROVENANCE]].eq(False).any(axis=1)
if bad.any():
    warnings.warn(f"PROVENANCE PROBLEM in rows {list(PROVENANCE.index[bad])}: do not use these numbers before resolving it.")
else:
    print("provenance: no failed check")

## Score every prediction

`cellpose.metrics.average_precision` is run per image at all IoU thresholds, for every model and
dataset listed as scored above. A model that is still incomplete, such as one from a running
session, is not scored. The results go to `results/cellpose_eval_per_image.csv`, one row per
model × image × IoU threshold.

In [ ]:
recs = []
for rec in IMAGES.to_dict("records"):
    gt = read_gt(rec["gt"])
    for m in INCLUDED[rec["dataset"]]:
        if not has_pred(rec[m]):
            continue
        pred = read_labels(rec[m])
        assert pred.shape == gt.shape, f"{m} {rec['dataset']}/{rec['id']}: {pred.shape} vs GT {gt.shape}"
        ap, tp, fp, fn = metrics.average_precision(gt, pred, threshold=IOU_THRESHOLDS)
        for k, th in enumerate(IOU_THRESHOLDS):
            recs.append((m, rec["dataset"], rec["id"], rec["held_out"], th, ap[k], tp[k], fp[k], fn[k]))

EVAL = pd.DataFrame(recs, columns=["model", "dataset", "id", "held_out", "iou", "ap", "tp", "fp", "fn"])
EVAL[["tp", "fp", "fn"]] = EVAL[["tp", "fp", "fn"]].astype(int)
with np.errstate(divide="ignore", invalid="ignore"):
    EVAL["error_rate"] = (EVAL.fp + EVAL.fn) / (EVAL.tp + EVAL.fn)
EVAL["label"] = EVAL.model.map({m: c["label"] for m, c in MODELS.items()})
EVAL["edited"] = EVAL.model.map({m: c["edited"] for m, c in MODELS.items()})
EVAL["model"] = pd.Categorical(EVAL.model, categories=list(MODELS), ordered=True)
EVAL["dataset"] = pd.Categorical(EVAL.dataset, categories=list(DATASETS), ordered=True)
EVAL.to_csv(RESULTS / "cellpose_eval_per_image.csv", index=False)
print(f"{len(EVAL):,} rows -> {RESULTS / 'cellpose_eval_per_image.csv'}")


def subset_mask(df, subset):
    return {"all": np.ones(len(df), bool), "held_out": df.held_out.to_numpy(),
            "dev": ~df.held_out.to_numpy()}[subset]


def at_iou(df, th):
    return df[np.isclose(df.iou, th)]

## Summary tables

The first table gives means per image and total TP/FP/FN at IoU 0.5, for held-out images, dev
images and all images. The second table is the paired comparison of each original model with its
Claude-edited version on identical images. In it, Δ = edited − original. "improved", "worse" and
"same" count images by their change in AP@0.5, and *p* comes from a two-sided Wilcoxon
signed-rank test. As in the paper, images with zero difference are dropped from the test.

In [ ]:
def summarize(subset):
    e = EVAL[subset_mask(EVAL, subset)]
    g = ["dataset", "model"]
    e50 = at_iou(e, 0.5)
    out = e50.groupby(g, observed=True).agg(images=("id", "size"), AP50=("ap", "mean"),
                                            error_rate50=("error_rate", "mean"),
                                            TP=("tp", "sum"), FP=("fp", "sum"), FN=("fn", "sum"))
    out["AP75"] = at_iou(e, 0.75).groupby(g, observed=True).ap.mean()
    out["AP90"] = at_iou(e, 0.9).groupby(g, observed=True).ap.mean()
    out["mAP50_95"] = e.groupby(g, observed=True).ap.mean()
    out.insert(0, "subset", subset)
    return out[["subset", "images", "AP50", "AP75", "AP90", "mAP50_95", "error_rate50", "TP", "FP", "FN"]]


SUMMARY = pd.concat([summarize(s) for s in ["held_out", "dev", "all"]])
SUMMARY.round(4).to_csv(RESULTS / "cellpose_summary.csv")
display(SUMMARY.round(4))


def paired_p(a, b):
    """Two-sided Wilcoxon signed-rank p (paper: scipy default, zero differences dropped)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    if not np.any(a[ok] != b[ok]):
        return np.nan                     # identical on every image
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return wilcoxon(a[ok], b[ok]).pvalue


def stars(p):
    return "n.s." if not np.isfinite(p) or p > 0.05 else "*" if p >= 0.01 else "**" if p >= 0.001 else "***"


def pairs():
    """(original, edited) model keys per family."""
    for fam in FAMILIES:
        for mo in [m for m, c in MODELS.items() if c["family"] == fam and not c["edited"]]:
            for me in [m for m, c in MODELS.items() if c["family"] == fam and c["edited"]]:
                yield fam, mo, me


def paired_table(subset, iou=IOU_MAIN):
    e = at_iou(EVAL[subset_mask(EVAL, subset)], iou)
    rows = []
    for fam, mo, me in pairs():
        for ds in DATASETS:
            a = e[(e.model == mo) & (e.dataset == ds)].set_index("id")
            b = e[(e.model == me) & (e.dataset == ds)].set_index("id")
            ids = a.index.intersection(b.index)
            if not len(ids):
                continue
            a, b = a.loc[ids], b.loc[ids]
            d = b.ap - a.ap
            rows.append(dict(
                family=fam, dataset=ds, subset=subset, images=len(ids),
                AP_original=a.ap.mean(), AP_claude=b.ap.mean(), dAP=d.mean(),
                improved=int((d > 0).sum()), worse=int((d < 0).sum()), same=int((d == 0).sum()),
                p_AP=paired_p(b.ap, a.ap),
                d_error_rate=(b.error_rate - a.error_rate).mean(), p_error=paired_p(b.error_rate, a.error_rate),
                dTP=int(b.tp.sum() - a.tp.sum()), dFP=int(b.fp.sum() - a.fp.sum()), dFN=int(b.fn.sum() - a.fn.sum()),
            ))
    return pd.DataFrame(rows)


PAIRED = pd.concat([paired_table(s) for s in ["held_out", "dev", "all"]], ignore_index=True)
PAIRED.to_csv(RESULTS / "cellpose_paired_tests.csv", index=False)
display(PAIRED.round({"AP_original": 4, "AP_claude": 4, "dAP": 4, "d_error_rate": 4}))

## Accuracy violin plots (Cellpose-SAM paper style)

The layout follows `supp_bench` in `paper/cpsam/figures.py`. The top row shows the per-image error
rate and the bottom row AP at IoU 0.5. Each group is one dataset, with one violin per model and a
thick line at the mean. Brackets compare each original model with its Claude-edited version
(paired Wilcoxon). Edited models are hatched.

In [ ]:
def style_violin(vp, color, edited):
    body = vp["bodies"][0]
    body.set_facecolor(mpl.colors.to_rgba(color, 0.35))
    body.set_edgecolor(color)
    body.set_linewidth(0.8)
    if edited:
        body.set_hatch("///")


def legend_handles(models):
    return [Patch(facecolor=mpl.colors.to_rgba(MODELS[m]["color"], 0.35), edgecolor=MODELS[m]["color"],
                  hatch="///" if MODELS[m]["edited"] else None, label=MODELS[m]["label"]) for m in models]


def violin_figure(subset, iou=IOU_MAIN, save=True):
    e = at_iou(EVAL[subset_mask(EVAL, subset)], iou)
    n_v = sum(len(INCLUDED[ds]) for ds in DATASETS)
    fig, axes = plt.subplots(2, 1, figsize=(1.6 + 0.75 * n_v + 0.6 * len(DATASETS), 7.2), sharex=True,
                             gridspec_kw=dict(hspace=0.45))
    metrics_rows = [("error_rate", f"error rate @ {iou:g} IoU"), ("ap", f"average precision (AP) @ {iou:g} IoU")]
    xticks, xlabels = [], []
    lims = {}
    for row, (ax, (col, ylabel)) in enumerate(zip(axes, metrics_rows)):
        axin = ax.inset_axes([0, 1.0, 1, 0.16 if row else 0.32])
        k, pos = 0.0, {}
        for ds in DATASETS:
            ms = INCLUDED[ds]
            if not ms:
                continue
            start = k
            for m in ms:
                vals = e[(e.model == m) & (e.dataset == ds)][col].to_numpy(float)
                vals = vals[np.isfinite(vals)]
                c = MODELS[m]["color"]
                if np.ptp(vals) > 0:
                    style_violin(ax.violinplot(vals, positions=[k], widths=0.6, showmeans=False,
                                               showextrema=False), c, MODELS[m]["edited"])
                ax.plot(k + 0.3 * np.array([-1, 1]), vals.mean() * np.ones(2), color=c, lw=3, solid_capstyle="butt")
                lims.setdefault(col, []).append((vals.min(), vals.max()))
                pos[(ds, m)] = k
                if row == 0:
                    xticks.append(k)
                    xlabels.append(MODELS[m]["label"])
                k += 1
            # original vs edited, paired on identical images
            for fam, mo, me in pairs():
                if (ds, mo) in pos and (ds, me) in pos:
                    a = e[(e.model == mo) & (e.dataset == ds)].set_index("id")[col]
                    b = e[(e.model == me) & (e.dataset == ds)].set_index("id")[col]
                    ids = a.index.intersection(b.index)
                    p = paired_p(b.loc[ids], a.loc[ids])
                    x0, x1 = pos[(ds, mo)], pos[(ds, me)]
                    y = 0.15 if row else 0.08
                    h = 0.25 if row else 0.12
                    axin.plot([x0, x0, x1, x1], [y, y + h, y + h, y], lw=1, color="k")
                    axin.text((x0 + x1) / 2, y + h + 0.02, stars(p), ha="center", va="bottom",
                              fontsize="small" if stars(p) == "n.s." else "large")
            if row == 0:
                n = len(e[(e.model == ms[0]) & (e.dataset == ds)])
                axin.text((start + k - 1) / 2, 0.62, f"{DATASETS[ds]}\nn={n:,}", ha="center", va="bottom",
                          fontsize="large")
            k += 1.5
        ax.set_ylabel(ylabel)
        ax.spines["bottom"].set_visible(False)
        ax.grid(True, color=0.8 * np.ones(3), lw=0.5, ls="--", axis="y")
        ax.set_axisbelow(True)
        ax.set_xlim(-0.6, k - 1.5 + 0.6)
        axin.set_xlim(ax.get_xlim())
        axin.set_ylim(0, 1)
        axin.axis("off")
    lo = min(v[0] for v in lims["ap"])
    hi = max(v[1] for v in lims["error_rate"])
    axes[1].set_ylim(min(0.4, np.floor(lo * 10) / 10), 1.0)
    axes[0].set_ylim(0, max(0.8, np.ceil(hi * 10) / 10))
    axes[1].set_xticks(xticks)
    axes[1].set_xticklabels(xlabels, rotation=45, ha="right")
    for ax in axes:
        ax.tick_params(axis="x", length=0)
    title = {"held_out": "held-out images (in no editing session's dev split)", "all": "all images",
             "dev": "dev images only (inspected while editing)"}[subset]
    fig.suptitle(f"Performance on the Cellpose datasets: {title}", x=0.02, ha="left", y=1.04,
                 fontstyle="italic", fontsize="large")
    plotted = [m for m in MODELS if any(m in INCLUDED[ds] for ds in DATASETS)]
    axes[0].legend(handles=legend_handles(plotted), loc="upper left", bbox_to_anchor=(1.01, 1.0),
                   frameon=False, fontsize="small")
    if save:
        for ext in ("pdf", "png"):
            fig.savefig(FIGURES / f"cellpose_violins_{subset}.{ext}")
    plt.show()


for subset in SUBSETS:
    violin_figure(subset)

## AP across IoU thresholds

The top row shows mean AP against IoU threshold, as in the Cellpose papers; edited models are
dashed. The original and edited curves are almost on top of each other, so the bottom row plots
their difference: mean ΔAP (edited − original) on the same images.

In [ ]:
def ap_curve_figure(subset="held_out", save=True):
    e = EVAL[subset_mask(EVAL, subset)]
    dss = [ds for ds in DATASETS if INCLUDED[ds]]
    fig, axes = plt.subplots(2, len(dss), figsize=(3.6 * len(dss) + 2.2, 6.4), squeeze=False,
                             gridspec_kw=dict(height_ratios=[1.4, 1], hspace=0.4))
    for ax in axes[1, 1:]:
        ax.sharey(axes[1, 0])
    for j, ds in enumerate(dss):
        ax = axes[0, j]
        for m in INCLUDED[ds]:
            g = e[(e.model == m) & (e.dataset == ds)].groupby("iou").ap.mean()
            ax.plot(g.index, g.values, color=MODELS[m]["color"], lw=2, ls="--" if MODELS[m]["edited"] else "-",
                    label=MODELS[m]["label"])
        ax.set_title(DATASETS[ds])
        ax.set_ylim(0, 1)
        ax.set_xlabel("IoU threshold")
        ax.grid(True, color=0.85 * np.ones(3), lw=0.5, ls="--")
        ax = axes[1, j]
        ax.axhline(0, color="0.4", lw=1)
        for fam, mo, me in pairs():
            if mo in INCLUDED[ds] and me in INCLUDED[ds]:
                a = e[(e.model == mo) & (e.dataset == ds)].set_index(["id", "iou"]).ap
                b = e[(e.model == me) & (e.dataset == ds)].set_index(["id", "iou"]).ap
                d = (b - a).dropna().groupby(level="iou").mean()
                ax.plot(d.index, d.values, color=MODELS[me]["color"], lw=2, marker="o", ms=4,
                        label=f"{MODELS[me]['label']} − {MODELS[mo]['label']}")
        ax.set_xlabel("IoU threshold")
        ax.grid(True, color=0.85 * np.ones(3), lw=0.5, ls="--")
        if j == 0:
            axes[0, j].set_ylabel("mean AP")
            ax.set_ylabel("mean ΔAP (edited − original)")
    axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.03, 1), frameon=False, fontsize="small")
    axes[1, -1].legend(loc="upper left", bbox_to_anchor=(1.03, 1), frameon=False, fontsize="small")
    fig.suptitle(f"AP vs. IoU threshold ({subset.replace('_', '-')} images)", x=0.02, ha="left",
                 fontstyle="italic", fontsize="large")
    if save:
        for ext in ("pdf", "png"):
            fig.savefig(FIGURES / f"cellpose_ap_vs_iou_{subset}.{ext}")
    plt.show()


ap_curve_figure("held_out")

## Segmentations side by side with the ground truth

Each row is one image, with these panels: the image, the ground truth, and then one panel per
model. In the model panels:

* **cyan**: predicted mask matched to a GT mask at IoU ≥ 0.5 (true positive)
* **red**: predicted mask with no match (false positive)
* **dashed yellow**: GT mask that no prediction matched (false negative)

The matching is the same assignment `cellpose.metrics.average_precision` uses, and the panels
assert that their counts agree with it.

`show_segmentations(dataset, ids, zoom=..., focus=(original, edited))` can be called on any
image. With `zoom` set to a window size in pixels, the window centres on the area where the two
`focus` models disagree most. The edits change only a few masks per image, so the zoomed view is
usually the informative one.

In [ ]:
GT_COLOR = (0.8, 0.8, 0.3)    # paper's ground-truth outline colour
TP_COLOR = "#00e5ff"
FP_COLOR = "#ff3b30"


def load_image(dataset, id_):
    """RGB image for display, each channel scaled to its 1st-99th percentile."""
    img = np.asarray(Image.open(IMAGE_ROOT / dataset / f"{id_}_img.png")).astype(np.float32)
    img = img[..., None] if img.ndim == 2 else img[..., :3]
    out = np.zeros(img.shape[:2] + (3,), np.float32)
    for c in range(img.shape[-1]):
        lo, hi = np.percentile(img[..., c], [1, 99])
        if hi > lo:
            out[..., c] = np.clip((img[..., c] - lo) / (hi - lo), 0, 1)
    if img.shape[-1] == 1:
        out[:] = out[..., :1]
    return out


def outlines(labels, keep=None):
    """Closed outline (x, y) of each mask label in `keep` (all labels if None)."""
    out = []
    for lab, sl in enumerate(find_objects(labels), start=1):
        if sl is None or (keep is not None and lab not in keep):
            continue
        sl = tuple(slice(max(s.start - 1, 0), s.stop + 1) for s in sl)
        cs, _ = cv2.findContours((labels[sl] == lab).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if cs:
            c = max(cs, key=len)[:, 0, :] + [sl[1].start, sl[0].start]
            out.append(np.vstack([c, c[:1]]))
    return out


def draw(ax, labels, color, keep=None, ls="-", lw=0.9):
    for c in outlines(labels, keep):
        ax.plot(c[:, 0], c[:, 1], color=color, lw=lw, ls=ls)


def match(gt, pred, th=0.5):
    """Matched (GT labels, predicted labels): same assignment as cellpose.metrics._true_positive."""
    if gt.max() == 0 or pred.max() == 0:
        return set(), set()
    iou = metrics._intersection_over_union(gt, pred)[1:, 1:]
    costs = -(iou >= th).astype(float) - iou / (2 * min(iou.shape))
    ti, pi = linear_sum_assignment(costs)
    ok = iou[ti, pi] >= th
    return set(ti[ok] + 1), set(pi[ok] + 1)


def window(shape, size, a=None, b=None):
    """(y0, y1, x0, x1): full image, or a size x size window where masks a and b differ most."""
    H, W = shape
    if size is None or size >= max(H, W):
        return 0, H, 0, W
    if a is not None and b is not None and np.any((a > 0) != (b > 0)):
        dens = uniform_filter(((a > 0) != (b > 0)).astype(np.float32), size=size, mode="constant")
        cy, cx = np.unravel_index(np.argmax(dens), dens.shape)
    else:
        cy, cx = H // 2, W // 2
    y0 = int(np.clip(cy - size // 2, 0, max(H - size, 0)))
    x0 = int(np.clip(cx - size // 2, 0, max(W - size, 0)))
    return y0, min(y0 + size, H), x0, min(x0 + size, W)


def show_segmentations(dataset, ids, models=None, zoom=None, focus=None, iou=0.5, notes=None, save=None):
    """Rows = images; panels = image | ground truth | one per model. notes: {id: text} for row titles."""
    recs = {r["id"]: r for r in IMAGES[IMAGES.dataset == dataset].to_dict("records")}
    models = models or [m for m in INCLUDED[dataset] if all(has_pred(recs[i][m]) for i in ids)]
    rows = []
    for id_ in ids:
        rec = recs[id_]
        gt = read_gt(rec["gt"])
        preds = {m: read_labels(rec[m]) for m in models}
        fa, fb = (read_labels(rec[focus[0]]), read_labels(rec[focus[1]])) if focus else (None, None)
        rows.append((id_, rec, gt, preds, window(gt.shape, zoom, fa, fb)))
    ncol = 2 + len(models)
    ratios = [(w[1] - w[0]) / (w[3] - w[2]) for *_, w in rows]
    fig, axes = plt.subplots(len(rows), ncol, figsize=(2.8 * ncol, sum(2.8 * r + 0.75 for r in ratios)),
                             squeeze=False, gridspec_kw=dict(height_ratios=ratios))
    for r, (id_, rec, gt, preds, (y0, y1, x0, x1)) in enumerate(rows):
        img = load_image(dataset, id_)
        sl = (slice(y0, y1), slice(x0, x1))
        tag = "held-out" if rec["held_out"] else "dev, seen while editing"
        note = f"   {notes[id_]}" if notes and id_ in notes else ""
        panels = [("image", None), ("ground truth", None)] + [(MODELS[m]["label"], m) for m in models]
        for c, (title, m) in enumerate(panels):
            ax = axes[r, c]
            ax.imshow(img[sl], interpolation="nearest")
            ax.set_xticks([])
            ax.set_yticks([])
            for s_ in ax.spines.values():
                s_.set_visible(False)
            if c == 0:
                ax.set_title(f"{dataset}/{id_} ({tag}){note}", fontsize="small")
                if zoom is not None:
                    ax.set_xlabel(f"x {x0}-{x1}, y {y0}-{y1}", fontsize="x-small")
                continue
            if m is None:
                draw(ax, gt[sl], GT_COLOR)
                ax.set_title(title, fontsize="small")
                continue
            mg, mp = match(gt, preds[m], iou)
            row = EVAL[(EVAL.model == m) & (EVAL.dataset == dataset) & (EVAL.id == id_) & np.isclose(EVAL.iou, iou)].iloc[0]
            assert len(mp) == row.tp, f"matching disagrees with cellpose.metrics for {m} {dataset}/{id_}"
            all_p, all_g = set(np.unique(preds[m])[1:]), set(np.unique(gt)[1:])
            draw(ax, gt[sl], GT_COLOR, keep=all_g - mg, ls=(0, (2, 1.5)), lw=1.0)
            draw(ax, preds[m][sl], TP_COLOR, keep=mp)
            draw(ax, preds[m][sl], FP_COLOR, keep=all_p - mp, lw=1.2)
            ax.set_title(title, fontsize="small")
            ax.set_xlabel(f"AP@{iou:g} {row.ap:.3f}   TP {row.tp}  FP {row.fp}  FN {row.fn}", fontsize="x-small")
    handles = [Line2D([], [], color=GT_COLOR, lw=1.5, label="ground truth"),
               Line2D([], [], color=TP_COLOR, lw=1.5, label=f"prediction matched (IoU ≥ {iou:g})"),
               Line2D([], [], color=FP_COLOR, lw=1.5, label="prediction unmatched (FP)"),
               Line2D([], [], color=GT_COLOR, lw=1.5, ls=(0, (2, 1.5)), label="ground truth missed (FN)")]
    fig.tight_layout(rect=(0, 0.6 / fig.get_figheight(), 1, 1))
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize="small")
    if save:
        for ext in ("pdf", "png"):
            fig.savefig(FIGURES / f"{save}.{ext}")
    plt.show()


def delta_ap(dataset, original, edited, subset="held_out", iou=0.5):
    e = at_iou(EVAL[subset_mask(EVAL, subset) & (EVAL.dataset == dataset)], iou)
    a = e[e.model == original].set_index("id").ap
    b = e[e.model == edited].set_index("id").ap
    return (b - a).dropna().sort_values()

### Images the edit changed most (held-out test images)

For each model family with both models available, this shows the three held-out test images
whose AP@0.5 improved most and the two that got worse most, if any. Set `EXAMPLES` to choose
images by hand.

In [ ]:
EXAMPLE_DATASET = "test"
EXAMPLES = {}            # e.g. {"cpsam": ["013", "016"]}; families left out are picked automatically
ZOOM = 192               # px window for the zoomed view

for fam, mo, me in pairs():
    if not (mo in INCLUDED[EXAMPLE_DATASET] and me in INCLUDED[EXAMPLE_DATASET]):
        print(f"{fam}: no complete original + edited predictions on {EXAMPLE_DATASET}, skipped")
        continue
    d = delta_ap(EXAMPLE_DATASET, mo, me)
    print(f"{MODELS[me]['label']} vs {MODELS[mo]['label']} on held-out {EXAMPLE_DATASET} images: "
          f"{(d > 0).sum()} improved, {(d < 0).sum()} worse, {(d == 0).sum()} unchanged (AP@0.5)")
    ids = EXAMPLES.get(fam) or list(d[d > 0].sort_values(ascending=False).index[:3]) + list(d[d < 0].index[:2])
    display(d.loc[ids].rename("ΔAP@0.5").to_frame().T.round(4))
    notes = {i: f"ΔAP@0.5 {d[i]:+.3f}" for i in ids}
    show_segmentations(EXAMPLE_DATASET, ids, notes=notes, save=f"cellpose_examples_{fam}_{EXAMPLE_DATASET}_full")
    show_segmentations(EXAMPLE_DATASET, ids, zoom=ZOOM, focus=(mo, me), notes=notes,
                       save=f"cellpose_examples_{fam}_{EXAMPLE_DATASET}_zoom")

### Any image

The examples in this cell are dense test images that the cpsam edit report names.

In [ ]:
show_segmentations("test", ["014", "027"], zoom=256, focus=("cpsam", "cpsam_CLAUDE"))

## Notes for interpreting the numbers

* **Not directly comparable with the paper's figures.** The paper's "Cellpose test set" is a
  67-image subset of the `images_cyto2` test split (`load_dataset` in `paper/cpsam/benchmarks.py`).
  Here `test` is the 68-image Cellpose test set.
* **Inference settings differ from the benchmark code.** The paper's benchmark code also used
  `cellprob_threshold=-0.5`, and for cyto3 it used ground-truth diameters with test-time
  augmentation. The sandbox runs used the label-free settings from the paper's Methods text
  instead: cellprob 0.0, flow 0.4, no diameter estimation for Cellpose-SAM, and the repo defaults
  for cyto3. The settings of each export are in its `manifest.json`.
* **Training overlap.** `train` and `train_cyto2` are training images, so differences there show
  how the edit behaves in-distribution, not how well the model generalizes.
* **Ground-truth encoding.** 11 `train_cyto2` GT files (744–758) store background as 65535 and use
  label 0 for a cell. In them, 65535 covers 20–97 % of the image with little predicted foreground,
  while the label-0 region is almost entirely predicted foreground. With `GT_BACKGROUND_65535`
  they are remapped; without the remapping, the background would count as one giant missed cell.
  The test set is not affected.
* **Held-out sets differ by family.** Cellpose-SAM's dev split contains 18 test images. The cyto3
  session never saw the test set. Because `held_out` excludes the dev images of every session, all
  models are compared on the same images.